# 01 - Create And Publish Prompt Dataset

            This notebook builds the controlled prompt bank for the OCN study, saves it to Google Drive, logs summary plots to W&B, and publishes it to Hugging Face.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, textwrap

def find_repo_root():
    try:
        import google.colab  # type: ignore  # noqa: F401
        from google.colab import drive  # type: ignore
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive")
    except Exception:
        pass

    candidates = [
        Path.cwd(),
        Path("/content/empty-negations"),
        Path("/content/drive/MyDrive/ocn_empty_negations"),
        Path("/content/drive/MyDrive/AutoRegressive-Bhasha/empty-negations"),
    ]
    for candidate in candidates:
        if (candidate / "src/ocn").exists():
            return candidate
    repo_url = os.environ.get("OCN_REPO_URL", "")
    if repo_url:
        target = Path("/content/empty-negations")
        if not target.exists():
            subprocess.run(["git", "clone", repo_url, str(target)], check=True)
        return target
    raise FileNotFoundError(
        "Could not find the empty-negations repo. Run this notebook from the repo, "
        "copy it to /content/drive/MyDrive/ocn_empty_negations, or set OCN_REPO_URL."
    )

REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "src"))
print("Repo:", REPO_ROOT)

In [ ]:
import json
            from pathlib import Path
            import matplotlib.pyplot as plt
            import seaborn as sns
            import wandb

            from ocn.colab_utils import login_huggingface, login_wandb, make_colab_paths, publish_dataframe_to_hf, save_dataframe
            from ocn.prompt_factory import build_prompt_dataset

            paths = make_colab_paths()
            config = json.loads((paths.project_root / "ocn_colab_config.json").read_text())
            login_huggingface("HF_WRITE_ACCESS")
            run = login_wandb(project="ocn-empty-negations", name=f"prompts-{config['run_id']}", config=config)
            sns.set_theme(style="whitegrid")

In [ ]:
prompts = build_prompt_dataset()
            prompts.head()

In [ ]:
local_csv = save_dataframe(prompts, Path(config["drive_data_root"]) / "ocn_prompts.csv")
            local_parquet = save_dataframe(prompts, Path(config["drive_data_root"]) / "ocn_prompts.parquet")

            repo_url = publish_dataframe_to_hf(
                prompts,
                repo_id=config["hf_prompt_repo"],
                split="train",
                private=config["hf_private"],
                card_path=REPO_ROOT / "dataset_cards/ocn_prompts.md",
                commit_message=f"Publish OCN prompts {config['run_id']}",
            )
            print("Saved:", local_csv)
            print("Published:", repo_url)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
            prompts["category"].value_counts().sort_values().plot(kind="barh", ax=axes[0], color="#4c78a8")
            axes[0].set_title("Prompts by category")
            axes[0].set_xlabel("count")
            prompts["variant"].value_counts().sort_values().plot(kind="barh", ax=axes[1], color="#f58518")
            axes[1].set_title("Prompts by variant")
            axes[1].set_xlabel("count")
            plt.tight_layout()

            figure_path = Path(config["drive_figure_root"]) / "01_prompt_dataset_counts.png"
            fig.savefig(figure_path, dpi=180, bbox_inches="tight")
            wandb.log({
                "prompt_count": len(prompts),
                "category_count": prompts["category"].nunique(),
                "variant_count": prompts["variant"].nunique(),
                "prompt_dataset_counts": wandb.Image(str(figure_path)),
                "prompt_table": wandb.Table(dataframe=prompts.head(200)),
            })
            run.finish()
            figure_path